# 1. The Engineering Problem: Function Approximation

In classic RL, if you want to know the value of an action, you look up $Q(s,a)$ in an array. In Deep RL, we parameterize the Value or Policy function with a neural network containing weights $\theta$.

Instead of an array lookup, your agent is doing a standard forward pass in PyTorch. The network takes the state $S_t$ as an input tensor and outputs either value estimates or action probabilities.

The core engineering challenge in Deep RL is **non-stationarity**. In supervised learning, your dataset is static and explicitly labeled. In RL, the agent generates its own training data as it interacts with the environment. If the agent's policy changes, the distribution of the data it collects changes entirely. This causes catastrophic forgetting and wildly unstable training.

# 2. Scaling Value-Based Methods: Deep Q-Networks (DQN)

If we take the Q-Learning concept from Lesson 1.3 and scale it, we get DQN. The neural network $Q_\theta(s, a)$ takes a state tensor and outputs a vector of Q-values (one for each possible action).

To train this network, we want to minimize the Mean Squared Error between our current Q-value prediction and the target Q-value (the immediate reward plus the discounted max Q-value of the next state):
$$L(\theta) = \mathbb{E} \left[ \left( R_{t+1} + \gamma \max_{a'} Q_\theta(S_{t+1}, a') - Q_\theta(S_t, A_t) \right)^2 \right]$$

### The Implementation Reality (Fixing the Instability):

If you just compute this loss and call `loss.backward()`, the network will likely collapse. To make this work in a real system, you must implement two architectural components:

1.  **Experience Replay Buffer:** You never train on the exact sequence of data the agent just collected, because sequential states ($S_t$, $S_{t+1}$) are highly correlated. Instead, the agent interacts with the environment and stores tuples of $(S_t, A_t, R_{t+1}, S_{t+1})$ in a massive FIFO queue (the buffer). Your PyTorch `DataLoader` then samples random mini-batches from this buffer to break the correlation.
2.  **Target Networks:** Look at the loss function above. You are using $Q_\theta$ to calculate both your prediction and your target. This is like trying to hit a moving dartboard. In production, you instantiate a second, frozen copy of your network called the **Target Network** ($Q_{\theta^-}$). You use the frozen network to calculate the target, and every few thousand steps, you hard-copy or soft-update the weights from the main network to the target network.

# 3. Scaling Policy-Based Methods: Actor-Critic & PPO

While DQN is great for discrete action spaces (like moving a paddle in OpenSpiel), it fails in continuous spaces or highly complex domains like language generation.

The modern industry standard for production RL (and the foundation of training LLMs) relies on **Actor-Critic** architectures, specifically algorithms like **PPO (Proximal Policy Optimization)**.

You are training two separate heads (or entirely separate networks):
-   **The Actor ($\pi_\theta(a|s)$):** Takes the state tensor and outputs a probability distribution over the action space (often using a Softmax layer).
-   **The Critic ($V_\phi(s)$):** Takes the state tensor and outputs a single scalar value estimating how good it is to be in that state.

### The Implementation Reality (Advantage Estimation):

How do we update the Actor's weights? We use the Critic to calculate the **Advantage ($A_t$)**. The Advantage tells us: Did taking this specific action result in a better or worse return than we generally expected from this state?
$$A_t = G_t - V_\phi(S_t)$$

If $A_t$ is positive, the action was better than expected, and we calculate a gradient that pushes the Actor's logits higher for that action. If $A_t$ is negative, we penalize it.

PPO introduces a specialized objective function that "clips" the gradient. If the Actor network discovers a great action, standard policy gradients might take a massive optimization step that destroys the network weights. PPO mathematically bounds the update so the new policy cannot deviate too far from the old policy in a single training step, ensuring monotonic, stable improvement.

# 4. The Architecture Bridge: Transformers as Agents

If you are comfortable building Transformer models from scratch—implementing your own positional embeddings, multi-head attention blocks, and encoder/decoders—the leap to Deep RL is conceptually very small.

In a modern Deep RL system:
-   **The State ($S_t$):** is a sequence of tokens (e.g., text, code, or flattened patches of an image).
-   **The Neural Network:** is your Transformer stack processing that sequence.
-   **The Actor Output:** is just the standard language modeling head projecting to your vocabulary size. The action space $\mathcal{A}$ is simply the set of all possible next tokens.
-   **The Environment:** is the system processing that token. (In OpenEnv, this means your PyTorch script sends the chosen token across the HTTP client to the isolated container, which evaluates the code or advances the game, and returns the next token/state and a reward).

# RLHF

A standard pre-trained LLM is just a highly sophisticated autocomplete engine. If you prompt it with "Write a Python script," it might respond with "to scrape a website" because it’s just continuing the internet’s most likely text.

**RLHF (Reinforcement Learning from Human Feedback)** is the pipeline we use to "align" that base model into an instruction-following assistant. In the context of our RL framework from Module 1:
- **The Agent:** is the LLM.
- **The Challenge:** We can't put a human in the environment loop to reward the agent in real-time. The agent generates tokens in milliseconds; humans take minutes to read them.

## Phase 1: Supervised Fine-Tuning (SFT)

Before we touch RL, we need a baseline policy that somewhat knows what it's doing. If you start PPO on a raw base model, the action space (the entire vocabulary) is too vast, and the agent will never randomly stumble upon a good, helpful paragraph.

- **The Data:** You hire human domain experts to write perfect prompt-response pairs.
- **The Training:** You fine-tune the base model on this data using standard next-token prediction (Cross-Entropy Loss).
- **The Output:** The **SFT Model**. It can follow instructions, but it's limited by the exact data it was trained on and hasn't learned to generalize a "preference" for good answers.

## Phase 2: Training the Reward Model (The Environment Proxy)

Since we can't use humans in the RL loop, we train a neural network to act as a simulated human. This is your **Reward Model (RM)**.

1. **Data Collection:** You take the SFT model, feed it a prompt, and sample 2 to 4 different responses.
2. **Human Ranking:** Humans read the generated responses and rank them from best to worst.
3. **Architecture:** You take an LLM (often the SFT model itself), strip off the language modeling head (the vocab projection), and replace it with a linear regression head that outputs a single scalar value.
4. **The Objective:** You train this model using a contrastive loss (like the Bradley-Terry model). If response $A$ is better than response $B$, you update the weights so that $RM(A) > RM(B)$.

You now have an incredibly fast, automated mathematical function that can score text exactly how a human would. This RM is your RL environment.

## Phase 3: The PPO Optimization Loop

This is where the systems engineering gets incredibly heavy. To run PPO on an LLM, you actually need to load four separate models into VRAM simultaneously:

- **The Actor Model ($\pi_\phi$):** The model being trained. It starts as a copy of the SFT model.
- **The Reference Model ($\pi_{\text{ref}}$):** A frozen copy of the SFT model.
- **The Critic Model ($V_\theta$):** A value network that predicts what score the Reward Model will give a specific sequence.
- **The Reward Model (RM):** The frozen proxy environment we built in Phase 2.

### The Production Loop:

1. **Rollout:** A prompt $x$ is fed to the Actor. The Actor generates a response $y$.
2. **Reward Calculation:** The Reward Model scores the response.
3. **The KL-Divergence Hack (Crucial):** If you just train the Actor to maximize the RM score, it will find a loophole (**Reward Hacking**). It will figure out that generating weird, repetitive phrases tricks the RM into giving a score of +100. To prevent this, we calculate the Kullback-Leibler (KL) divergence between the Actor's output probabilities and the frozen Reference Model's probabilities.
4. **The Modified Reward:**
   $$R(x, y) = \text{RM\_Score}(x, y) - \beta \log \frac{\pi_\phi(y|x)}{\pi_{\text{ref}}(y|x)}$$
   *(Translation: Maximize the human preference score, but if you deviate too far from the original human-written SFT style, you get penalized. $\beta$ is the penalty coefficient.)*
5. **PPO Update:** The Critic calculates the Advantage, and the Actor is updated using the clipped PPO loss we discussed in Lesson 2.1.

### The Hardware Reality

The main reason RLHF is hard to implement is memory. If you are aligning a 70B parameter model, holding four copies of it (or variants of it) in memory requires massive compute clusters utilizing DeepSpeed or FSDP (Fully Sharded Data Parallelism).

This memory bottleneck is exactly why the industry is currently shifting toward newer algorithms that drop the Critic and Reward models entirely.

# Next-Gen Algorithms (GRPO and the End of the Critic)

## 1. The PPO Memory Wall

In standard PPO, the Advantage ($A_t$) tells the Actor whether an action was better or worse than expected. To calculate that expectation, PPO requires a completely separate neural network: **The Critic**.

If you are training an 8-billion parameter LLM, your Critic is usually another 8-billion parameter model. As models scale to 70B or 400B parameters, the VRAM requirements for holding a Critic model become financially and architecturally catastrophic.

## 2. The GRPO Solution: Dropping the Critic

**GRPO (Group Relative Policy Optimization)** was designed to solve this exact hardware bottleneck. The core innovation of GRPO is incredibly simple but mathematically elegant: **It completely eliminates the Critic model.**

Instead of relying on a massive, parameterized neural network to estimate a baseline value for a state, GRPO estimates the baseline dynamically by comparing multiple outputs generated from the exact same prompt.

## 3. The GRPO Architecture and Math

Here is exactly how the GRPO pipeline executes in a training loop:

1. **Group Sampling:** Given a single input prompt $x$, the current Actor model ($\pi_{\theta_{old}}$) generates a group of $G$ different responses: $\{y_1, y_2, ..., y_G\}$. (Usually, $G$ is between 4 and 8).
2. **Reward Scoring:** Each response is evaluated by the Reward Model (or a rule-based environment, like a code compiler or a math verifier) to get a set of raw reward scores: $\{r_1, r_2, ..., r_G\}$.
3. **Relative Advantage Calculation:** Because we have a group of responses to the exact same prompt, we don't need a Critic to tell us the "expected" value of the prompt. We just calculate the mean ($\mu$) and standard deviation ($\sigma$) of the rewards in our group. The Advantage ($A_i$) for each response is simply its normalized score (Z-score):
   $$A_i = \frac{r_i - \mu}{\sigma}$$
   If response $y_1$ scored higher than the group average, it gets a positive advantage. If $y_2$ was worse than average, it gets a negative advantage.

### The Objective Function:
The Actor's weights are then updated using a clipped surrogate objective very similar to PPO, alongside a KL divergence penalty (to ensure the model doesn't drift too far from the reference model). For each response $i$ in the group $G$:

$$\mathcal{J}_{GRPO}(\theta) = \mathbb{E}_{x, y \sim \pi_{\theta_{old}}} \left[ \frac{1}{G} \sum_{i=1}^G \min \left( \frac{\pi_\theta(y_i|x)}{\pi_{\theta_{old}}(y_i|x)} A_i, \text{clip}\left(\frac{\pi_\theta(y_i|x)}{\pi_{\theta_{old}}(y_i|x)}, 1-\epsilon, 1+\epsilon\right) A_i \right) - \beta \mathbb{D}_{KL} \right]$$

## 4. Why This Matters for Production Environments (Like OpenEnv)

By dropping the Critic, GRPO essentially cuts the VRAM training requirements in half. This is what allowed labs to scale RL to massive models, leading to the highly capable reasoning models we see today.

However, as the OpenEnv documentation points out, having an algorithm like GRPO is only half the battle: *"The problem still remains, how do you take these RL algorithms and take them beyond Cartpole?"*.

If you are training a GRPO agent to write code or play complex games, you need a safe, scalable way to execute those actions to generate the rewards ($r_1 ... r_G$). If the agent writes a Python script that formats a hard drive, you cannot execute that in your main PyTorch training thread. This is exactly why the architecture of **OpenEnv**—packaging the environment inside isolated Docker containers and exposing it via a type-safe HTTP API—is the necessary infrastructure to actually put GRPO into practice.

# RL as a sequence of tokens like normal text

# 1. The Paradigm Shift: RL as Sequence Modeling

In standard Deep RL, you are constantly fighting the environment's dynamics to estimate future value. In the **Decision Transformer** framework, we abandon Bellman equations entirely. We treat the entire history of an episode as a single sequence. If a language model can learn the grammar of English by reading billions of sentences, a transformer can learn the "grammar" of optimal decision-making by reading millions of gameplay trajectories.

## 2. The Data Structure: Return-to-Go (RTG)

To make this work, we have to feed the transformer a sequence. A standard RL trajectory looks like this:
$$State_1, Action_1, Reward_1, State_2, Action_2, Reward_2, \dots$$

But predicting the next action based on past rewards doesn't help the model learn to achieve future rewards. The core engineering trick of the Decision Transformer is replacing the immediate reward with the **Return-to-Go (RTG)**.

The RTG is the sum of all future rewards from that timestep onward. The trajectory we actually feed into the transformer is:
$$\tau = (\hat{R}_1, s_1, a_1, \hat{R}_2, s_2, a_2, \dots, \hat{R}_T, s_T, a_T)$$
Where $\hat{R}_t = \sum_{k=t}^T r_k$.

**Why do this?** Because it allows us to condition the transformer on the desired outcome.

## 3. The Architecture Stack

Building a Decision Transformer is virtually identical to building a small LLM (like GPT-2 or Llama), with a few modifications to the embedding layer.

### The Inputs:
At any timestep $t$, the transformer receives three tokens: the RTG ($\hat{R}_t$), the State ($s_t$), and the previous Action ($a_{t-1}$).

### Linear Embeddings:
Instead of a standard token embedding matrix (like `nn.Embedding` for text), we project these different modalities into the same hidden dimension ($d_{model}$) using linear layers:
- **State Embedding:** A linear layer (or CNN if the state is an image) mapping the state to a $d_{model}$ vector.
- **Action Embedding:** A linear layer mapping the action to a $d_{model}$ vector.
- **RTG Embedding:** A linear layer mapping the scalar RTG to a $d_{model}$ vector.

### Positional Encoding & Attention:
- **Timestep Embeddings:** Crucially, we use timestep embeddings rather than absolute positional embeddings. $\hat{R}_t$, $s_t$, and $a_t$ all receive the exact same timestep embedding so the self-attention mechanism knows they occurred simultaneously.
- **Causal Attention:** The sequence is passed through standard causal Transformer blocks (masked self-attention) so the model can only attend to past events, not the future.
- **The Output Head:** We only care about predicting the next action. We attach a linear projection head to the state embeddings in the final layer to predict $a_t$.

## 4. Training vs. Inference (The "Prompting" Hack)

### Training (Offline RL):
Training is shockingly simple. It is purely supervised learning. You take a massive dataset of recorded trajectories (some good, some bad), calculate the RTGs, and train the model using standard Cross-Entropy Loss (for discrete actions) or Mean Squared Error (for continuous actions) to predict the ground-truth action $a_t$ given the preceding sequence. There is no environment interaction during training.

### Inference (Online Execution):
This is where the magic happens. How do you get the model to perform well in the real environment? **You prompt it.**

When you start an episode, you feed the model the starting state $s_1$, and you artificially inject a high target RTG (e.g., $\hat{R}_1 = 100$, if 100 is the max possible score). 

Because the transformer learned the correlations between RTGs, states, and actions, "prompting" it with a high RTG forces it to autocomplete the sequence of actions that traditionally led to that high score. As the agent takes actions and receives actual rewards from the environment, you simply subtract the received reward from your target RTG and feed the updated RTG into the next timestep.

## 5. Integration with OpenEnv

This architecture maps flawlessly onto the OpenEnv production patterns you provided. Instead of dealing with PPO replay buffers, your system architecture looks like this:

1. **The Client:** You instantiate `env = OpenSpielEnv(...)`.
2. **The Context Window:** You maintain a rolling queue of `(RTG, Observation, Action)` in your Python script.
3. **The Forward Pass:** You pass this queue into your PyTorch Decision Transformer to predict the next `action_id`.
4. **The Environment Step:** You wrap that integer into an `OpenSpielAction` and send it over HTTP via `env.step(action)`.
5. **The Update:** You take the `result.reward`, subtract it from your target RTG, append the new `result.observation` to your context window, and repeat.

OpenEnv's type-safe `StepResult` handles all the JSON parsing and network latency behind the scenes, allowing your transformer to just consume a clean, typed stream of tokens.